In [ ]:
import pandas as pd
import numpy as np
import iqplot
from plot_tools import *
import itertools as it
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from scipy.spatial.distance import jensenshannon
hv.extension('bokeh')


In [ ]:
df_abun = pd.read_csv('e003_coalescence_metadata_round4_abundances.csv')
df_abun['AC']=df_abun['parent_subjects'].transform(lambda x: 'AC' in x)
df_abun=df_abun.loc[~df_abun['AC'],:]
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
bad_samples = df_abun_get_alpha.loc[df_abun_get_alpha['counts']<20,:]
bad_samples
df_abund = df_abun.loc[~df_abun['sample'].isin(bad_samples),:]
df_abun.head()

## looking at final timepoint

In [ ]:
df_abun_p7 = df_abun.loc[df_abun['passage'] == 7,:]
sample1s = []
sample2s = []
JSDs = []
for sample1,sample2 in it.combinations(df_abun_p7['sample'].unique(),2):
    sample1s.append(sample1)
    sample2s.append(sample2)
    s1_abuns = df_abun.loc[df_abun['sample'] == sample1,:].sort_values(by='species_id')
    s2_abuns  = df_abun.loc[df_abun['sample'] == sample2,:].sort_values(by='species_id')
    JSDs.append(jensenshannon(s1_abuns['relative_abundance'].values, s2_abuns['relative_abundance'].values))

In [ ]:
df_pairwise_JSD = pd.DataFrame(data = {'sample1': sample1s, 'sample2': sample2s, 
                                       'JSD': JSDs, })
df_pairwise_JSD['type_mesocosm1']=np.nan
df_pairwise_JSD['type_mesocosm2']=np.nan
for type_meso in df_abun['type_mesocosm'].unique():
    samples_meso=df_abun.loc[df_abun['type_mesocosm']==type_meso,'sample'].unique()
    df_pairwise_JSD.loc[df_pairwise_JSD['sample1'].isin(samples_meso),'type_mesocosm1']=type_meso
    df_pairwise_JSD.loc[df_pairwise_JSD['sample2'].isin(samples_meso),'type_mesocosm2']=type_meso

In [ ]:
df_pairwise_JSD['subjects1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_pairwise_JSD['subjects2'] =df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[:2]))

df_pairwise_JSD['env1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[2:]))
df_pairwise_JSD['env2'] = df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[2:]))

In [ ]:
df_pairwise_JSD['parent_media1']=df_pairwise_JSD['env1'].transform(lambda x: x.split('-')[0])
df_pairwise_JSD['parent_media2']=df_pairwise_JSD['env2'].transform(lambda x: x.split('-')[0])
df_pairwise_JSD['coal_media1']=df_pairwise_JSD['env1'].transform(lambda x: x.split('-')[1])
df_pairwise_JSD['coal_media2']=df_pairwise_JSD['env2'].transform(lambda x: x.split('-')[1])

In [ ]:
df_pairwise_JSD['same_subjects'] = df_pairwise_JSD['subjects1'] == df_pairwise_JSD['subjects2'] 
df_pairwise_JSD['same_coal_media'] = df_pairwise_JSD['coal_media1'] == df_pairwise_JSD['coal_media2'] 
df_pairwise_JSD['same_parent_media'] = df_pairwise_JSD['parent_media1'] == df_pairwise_JSD['parent_media2'] 
df_pairwise_JSD['same_env'] = df_pairwise_JSD['env1'] == df_pairwise_JSD['env2'] 
df_pairwise_JSD_only_double = df_pairwise_JSD#.loc[~df_pairwise_JSD['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                   #   'AE-AE','AF-AF']),:]

#df_pairwise_JSD_only_double = df_pairwise_JSD_only_double.loc[~df_pairwise_JSD_only_double['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
 #                                                                                     'AE-AE','AF-AF']),:]

In [ ]:
df_pairwise_JSD.to_csv('pairwise_JSD.csv')

In [ ]:
df_pairwise_JSD_only_double['JSD']

In [ ]:
df_pairwise_JSD_only_double['comparison']='Diff Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[df_pairwise_JSD_only_double['same_subjects'],'comparison']= 'Same Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Same Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Same Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Same Subject-Same Parent Media - Same Coal Media'

df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Diff Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Diff Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Diff Subject-Same Parent Media - Same Coal Media'

df_pairwise_JSD_only_double_double = df_pairwise_JSD_only_double.loc[~df_pairwise_JSD_only_double['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_double_double = df_pairwise_JSD_only_double_double.loc[~df_pairwise_JSD_only_double_double['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                     'AE-AE','AF-AF']),:]

p = iqplot.strip(df_pairwise_JSD_only_double_double, 
                 q = 'JSD', cats = ['comparison'], spread='jitter',palette = bokeh.palettes.Colorblind[8],
               #  palette = [bokeh.palettes.Colorblind[7][0],bokeh.palettes.Colorblind[7][4],
                #            bokeh.palettes.Colorblind[7][-2],bokeh.palettes.Colorblind[7][1],],
                width =650,height=500,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.xaxis.major_label_orientation = 4*np.pi/10
p.output_backend = 'svg'
export_plot_pdf(p, 'coal_JSD_media_subj')
p.xaxis.axis_label = 'Same Media - Same Subject'
bokeh.io.show(p)

In [ ]:

df_pairwise_JSD_only_double_single = df_pairwise_JSD_only_double.loc[df_pairwise_JSD_only_double['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_double_single = df_pairwise_JSD_only_double_single.loc[df_pairwise_JSD_only_double_single['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                     'AE-AE','AF-AF']),:]

p = iqplot.strip(df_pairwise_JSD_only_double_single.sort_values(by='comparison'), 
                 q = 'JSD', cats = ['comparison'], spread='jitter',palette = bokeh.palettes.Colorblind[8],#color_column='subjects1',
               #  palette = [bokeh.palettes.Colorblind[7][0],bokeh.palettes.Colorblind[7][4],
                #            bokeh.palettes.Colorblind[7][-2],bokeh.palettes.Colorblind[7][1],],
                width =650,height=500,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.xaxis.major_label_orientation = 4*np.pi/10
p.output_backend = 'svg'
export_plot_pdf(p, 'coal_JSD_media_subj_ss')
p.xaxis.axis_label = 'Same Media - Same Subject'
bokeh.io.show(p)

In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
c1s = []
c2s = []
pvalues = []
stats = []
for pairs in itertools.combinations(df_pairwise_JSD_only_double_double['comparison'].unique(),2):
    c1,c2 = pairs
    x = df_pairwise_JSD_only_double_double.loc[df_pairwise_JSD_only_double_double['comparison']==c1,'JSD'].values
    y = df_pairwise_JSD_only_double_double.loc[df_pairwise_JSD_only_double_double['comparison']==c2,'JSD'].values
    c1s.append(c1)
    c2s.append(c2)
    
    res = permutation_test((x, y), statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='two-sided')
    pvalues.append(res.pvalue)
    stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'compare1':c1s,'compare2':c2s,
                   'pval':pvalues ,'stats':stats,'qvals':qvals})
df['sig']=False
df.loc[df['qvals']<.05,'sig']='*'
df.loc[df['qvals']<.001,'sig']='**'
df.sort_values(by='qvals')


In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
c1s = []
c2s = []
pvalues = []
stats = []
for pairs in itertools.combinations(df_pairwise_JSD_only_double_single['comparison'].unique(),2):
    c1,c2 = pairs
    x = df_pairwise_JSD_only_double_single.loc[df_pairwise_JSD_only_double_single['comparison']==c1,'JSD'].values
    y = df_pairwise_JSD_only_double_single.loc[df_pairwise_JSD_only_double_single['comparison']==c2,'JSD'].values
    c1s.append(c1)
    c2s.append(c2)
    
    res = permutation_test((x, y), statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='two-sided')
    pvalues.append(res.pvalue)
    stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'compare1':c1s,'compare2':c2s,
                   'pval':pvalues ,'stats':stats,'qvals':qvals})
df['sig']=False
df.loc[df['qvals']<.05,'sig']='*'
df.loc[df['qvals']<.001,'sig']='**'
df.sort_values(by='qvals')
df.loc[(df['compare1']=='Same Subject-Same Parent Media - Same Coal Media')+(df['compare2']=='Same Subject-Same Parent Media - Same Coal Media'),
:]

In [ ]:
df['sig']=False
df.loc[df['qvals']<.05,'sig']='*'
df.loc[df['qvals']<.001,'sig']='**'
df.sort_values(by='qvals')


In [ ]:
df_pairwise_JSD_only_double['comparison']='Diff Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[df_pairwise_JSD_only_double['same_subjects'],'comparison']= 'Same Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Same Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Same Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Same Subject-Same Parent Media - Same Coal Media'

df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Diff Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Diff Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Diff Subject-Same Parent Media - Same Coal Media'

p = iqplot.strip(df_pairwise_JSD_only_double.sort_values(by='comparison'),
                 q = 'JSD', cats = ['comparison'], spread='jitter',palette = bokeh.palettes.Colorblind[8],
               #  palette = [bokeh.palettes.Colorblind[7][0],bokeh.palettes.Colorblind[7][4],
                #            bokeh.palettes.Colorblind[7][-2],bokeh.palettes.Colorblind[7][1],],
                width =650,height=500,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.xaxis.major_label_orientation = 4*np.pi/10
#Diff Subject-Same Parent Media - Same Coal Media	Same Subject-Diff Parent Media - Same Coal Media
dfsig = df.loc[df['sig']=='**',:].sort_values(by='compare2')
dfsig['place']=np.linspace(.85,.99,len(dfsig))

p.segment(source = dfsig, x0='compare1',x1='compare2',y1='place',y0='place',color='black')
bokeh.io.show(p)

In [ ]:
df.loc[df['sig']=='*',:]

## stability

In [ ]:
df_abun['parent_subjects'].unique()

In [ ]:

sample1s = []
sample2s = []
JSDs = []
for type_meso in df_abun['mesocosm'].unique():
    df_abun_type_meso = df_abun.loc[df_abun['mesocosm'] == type_meso,:]
    in_sample = df_abun_type_meso['inoculumn_sample'].values[0]
    good_samples = list(df_abun_type_meso['sample'].unique()) + [in_sample]
    for sample1,sample2 in it.combinations(good_samples,2):
        sample1s.append(sample1)
        sample2s.append(sample2)
        s1_abuns = df_abun.loc[df_abun['sample'] == sample1,:].sort_values(by='species_id')
        s2_abuns  = df_abun.loc[df_abun['sample'] == sample2,:].sort_values(by='species_id')
        JSDs.append(jensenshannon(s1_abuns['relative_abundance'].values, s2_abuns['relative_abundance'].values))

In [ ]:
df_pairwise_JSD = pd.DataFrame(data = {'sample1': sample1s, 'sample2': sample2s, 
                                       'JSD': JSDs, })

df_pairwise_JSD['type_mesocosm1'] = df_pairwise_JSD['sample1'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'type_mesocosm'].values[0])
df_pairwise_JSD['type_mesocosm2'] = df_pairwise_JSD['sample2'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'type_mesocosm'].values[0])

#df_pairwise_JSD['subject1'] = df_pairwise_JSD['sample1'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'subject'].values[0])
#df_pairwise_JSD['subject2'] = df_pairwise_JSD['sample2'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'subject'].values[0])

df_pairwise_JSD['passage1'] = df_pairwise_JSD['sample1'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'passage'].values[0])
df_pairwise_JSD['passage2'] = df_pairwise_JSD['sample2'].transform(lambda x: df_abun.loc[df_abun['sample'] == x,'passage'].values[0])

df_pairwise_JSD['passage1-passage2'] =  df_pairwise_JSD['passage1'].astype(str) + '-' + df_pairwise_JSD['passage2'].astype(str)

In [ ]:
df_pairwise_JSD['subjects1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_pairwise_JSD['subjects2'] =df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[:2]))

df_pairwise_JSD['env1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[2:]))
df_pairwise_JSD['env2'] = df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[2:]))

In [ ]:
df_pairwise_JSD['same_subjects'] = df_pairwise_JSD['subjects1'] == df_pairwise_JSD['subjects2'] 
df_pairwise_JSD['same_envs'] = df_pairwise_JSD['env1'] == df_pairwise_JSD['env2'] 
df_pairwise_JSD['passage1-passage2'] =  df_pairwise_JSD['passage1'].astype(str) + '-' + df_pairwise_JSD['passage2'].astype(str)
df_pairwise_JSD['passage1-passage2'] = df_pairwise_JSD['passage1-passage2'].transform(lambda x: '-'.join(np.sort(x.split('-'))))
df_pairwise_JSD['Media Switch']=False
df_pairwise_JSD.loc[df_pairwise_JSD['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'Media Switch']=True
df_pairwise_JSD['mswitch']='F'
df_pairwise_JSD.loc[df_pairwise_JSD['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'mswitch']='T'

df_pairwise_JSD_only_double = df_pairwise_JSD.loc[~df_pairwise_JSD['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_double = df_pairwise_JSD_only_double.loc[~df_pairwise_JSD_only_double['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]


df_pairwise_JSD_good = df_pairwise_JSD_only_double.loc[df_pairwise_JSD_only_double['passage1-passage2'].isin(['0-1','1-2','2-3','3-4','4-5','5-6','6-7']),:]

In [ ]:
df_pairwise_JSD_good=df_pairwise_JSD_good.loc[~df_pairwise_JSD_good['JSD'].isna(),:]
p = iqplot.strip(df_pairwise_JSD_good.sort_values(by='passage1-passage2', ascending=True), 
                 q = 'JSD', cats = ['passage1-passage2','mswitch' ], spread='jitter',
                 color_column = 'mswitch',
                 palette = bokeh.palettes.Colorblind[7][::-1],
                    width =650,height=200,show_legend=False,
                 q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
#p.xaxis.axis_label = 'Passage1-Passage2'
p.legend.location='right'
p.output_backend='svg'
export_plot_pdf(p,'overtime_jsd_coal')
bokeh.io.show(p)

In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
p1p2s=[]
pvalues = []
stats = []
for p1p2 in df_pairwise_JSD_good['passage1-passage2'].unique():
    p1p2_df =  df_pairwise_JSD_good.loc[df_pairwise_JSD_good['passage1-passage2']==p1p2,:]
    x=p1p2_df.loc[p1p2_df['mswitch']=='T','JSD'].values
    y=p1p2_df.loc[p1p2_df['mswitch']=='F','JSD'].values
    p1p2s.append(p1p2)
    
    res = permutation_test((x,y),statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='greater')
    pvalues.append(res.pvalue)
    stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'p1p2s':p1p2s,
                   'pval':pvalues ,'stats':stats,'qvals':qvals})
df.sort_values(by='qvals')


In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
midpassages = []
mswitchs=[]
pvalues = []
stats = []
for p1p2 in [('0-1','1-2'),('1-2','2-3'),('2-3','3-4'),('3-4','4-5'),('4-5','5-6'),('5-6','6-7')]:
    p1p21,p1p22= p1p2
    midpassage = p1p21[-1]
    

    
    for i in ['T','F']:
        midpassages.append(midpassage)
        mswitchs.append(i)
       
        p1p2_df =  df_pairwise_JSD_good.loc[df_pairwise_JSD_good['mswitch']==i,:]
 
        x=p1p2_df.loc[p1p2_df['passage1-passage2']==p1p21,'JSD'].values
        y=p1p2_df.loc[p1p2_df['passage1-passage2']==p1p22,'JSD'].values
    
        
        res = permutation_test((x,y),statistic, vectorized=True,permutation_type='independent',
                           n_resamples=10000, alternative='two-sided')
        pvalues.append(res.pvalue)
        stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'midpassages':midpassages, 'mswitchs':mswitchs,
                   'pval':pvalues ,'stats':stats,'qvals':qvals})
df.sort_values(by='qvals')

In [ ]:
df_pairwise_JSD_only_single = df_pairwise_JSD.loc[df_pairwise_JSD['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_single = df_pairwise_JSD_only_single.loc[df_pairwise_JSD_only_single['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]


df_pairwise_JSD_good = df_pairwise_JSD_only_single.loc[df_pairwise_JSD_only_single['passage1-passage2'].isin(['0-1','1-2','2-3','3-4','4-5','5-6','6-7']),:]
df_pairwise_JSD_good['Media Switch']=False
df_pairwise_JSD_good.loc[df_pairwise_JSD_good['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'Media Switch']=True
df_pairwise_JSD_good['mswitch']='F'
df_pairwise_JSD_good.loc[df_pairwise_JSD_good['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'mswitch']='T'
p = iqplot.strip(df_pairwise_JSD_good.sort_values(by='passage1-passage2', ascending=True),
                 q = 'JSD', cats = ['passage1-passage2','mswitch' ], spread='jitter',
                 color_column = 'Media Switch',
                 palette = [bokeh.palettes.Colorblind[7][0],bokeh.palettes.Colorblind[7][-1]],
                width = 600,
                 show_legend=True,q_axis='y')

p.ygrid.grid_line_color = None

p.y_range = bokeh.models.Range1d(0,1)
p.xaxis.axis_label = 'Passage1-Passage2'
bokeh.io.show(p)

In [ ]:
p=hv.Points(df_pairwise_JSD_good.sort_values(by='passage1-passage2'), 
            vdims=['passage1-passage2','Media Switch'],kdims=['passage1-passage2','JSD']).opts(color='Media Switch')
p

# number of sp

In [ ]:
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
df_abun_get_alpha_only_double = df_abun_get_alpha.loc[~df_abun_get_alpha['subjects'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]
df_abun_get_alpha_only_double.head()

In [ ]:
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
bad_samples = df_abun_get_alpha.loc[df_abun_get_alpha['counts']<20,:]
bad_samples

In [ ]:
p = iqplot.strip(df_pairwise_JSD_good.sort_values(by='passage1-passage2', ascending=True), 
                 q = 'JSD', cats = ['passage1-passage2','mswitch' ], spread='jitter',
                 color_column = 'mswitch',
                 palette = bokeh.palettes.Colorblind[7][::-1],
                    width =650,height=200,
                 show_legend=True,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.xaxis.axis_label = 'Passage1-Passage2'
p.legend.location='right'
p.output_backend='svg'
#export_plot_pdf(p,'overtime_single_JSD')
bokeh.io.show(p)

In [ ]:
df_pairwise_JSD_only_single = df_pairwise_JSD.loc[df_pairwise_JSD['subjects1'].isin(['AA-AA',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_single = df_pairwise_JSD_only_single.loc[df_pairwise_JSD_only_single['subjects2'].isin(['AA-AA',
                                                                                      'AE-AE','AF-AF']),:]


df_pairwise_JSD_good = df_pairwise_JSD_only_single.loc[df_pairwise_JSD_only_single['passage1-passage2'].isin(['0-1','1-2','2-3','3-4','4-5','5-6','6-7']),:]
df_pairwise_JSD_good['Media Switch']=False
df_pairwise_JSD_good.loc[df_pairwise_JSD_good['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'Media Switch']=True
df_pairwise_JSD_good['mswitch']='F'
df_pairwise_JSD_good.loc[df_pairwise_JSD_good['env1'].isin(['mGAM-mBHI','mBHI-mGAM']),'mswitch']='T'
p = iqplot.strip(df_pairwise_JSD_good.sort_values(by='passage1-passage2', ascending=True),
                 q = 'JSD', cats = ['passage1-passage2','mswitch' ], spread='jitter',
                 color_column = 'Media Switch',
                 palette =  bokeh.palettes.Colorblind[7][::-1],
               width =650,height=200,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None

p.y_range = bokeh.models.Range1d(0,1)
#p.xaxis.axis_label = 'Passage1-Passage2'
p.output_backend='svg'
export_plot_pdf(p,'overtime_single_JSD')
bokeh.io.show(p)


In [ ]:
df_abun_get_alpha_only_double['coalescence_media']=df_abun_get_alpha_only_double['type_mesocosm'].transform(lambda x: x.split('-')[-1])
df_abun_get_alpha_only_double['parent_media']=df_abun_get_alpha_only_double['type_mesocosm'].transform(lambda x: x.split('-')[-2])
df_abun_get_alpha_only_double['env']=df_abun_get_alpha_only_double['parent_media']+'-'+df_abun_get_alpha_only_double['coalescence_media']
df_abun_get_alpha_only_double['mswitch'] ='F'
df_abun_get_alpha_only_double.loc[df_abun_get_alpha_only_double['parent_media']!=df_abun_get_alpha_only_double['coalescence_media'],'mswitch']='T'

df_abun_get_alpha_only_double=df_abun_get_alpha_only_double.loc[df_abun_get_alpha_only_double['counts']>20,:]#.unique()
p = iqplot.strip(df_abun_get_alpha_only_double.sort_values(by='passage', ascending=True), 
                 q = 'counts', cats = ['passage','mswitch' ], spread='jitter',
                 color_column = 'mswitch',
                palette= bokeh.palettes.Colorblind[7][::-1],
                  width =650,height=200,show_legend=False,
                 q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,60)
#p.xaxis.axis_label = 'Same Media - Same Subject'
p.yaxis.axis_label = 'Number of Species'
p.legend.location='right'
p.output_backend='svg'
export_plot_pdf(p,'num_sp_overtime_coal')
bokeh.io.show(p)

In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
passages = []
pvalues = []
stats = []
for passage in [1,2,3,4,5,6,7]:
    passages.append(passage)
    p1p2_df =  df_abun_get_alpha_only_double.loc[df_abun_get_alpha_only_double['passage']==passage,:]
 
    x=p1p2_df.loc[p1p2_df['mswitch'],'counts'].values
    y=p1p2_df.loc[~p1p2_df['mswitch'],'counts'].values
    
    
        
    res = permutation_test((x,y),statistic, vectorized=True,permutation_type='independent',
                           n_resamples=10000, alternative='two-sided')
    pvalues.append(res.pvalue)
    stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'passages':passages, 
                   'pval':pvalues,'stats':stats,'qvals':qvals})
df.sort_values(by='qvals')
df['sigs']=df['qvals']<.05
df['sigsig']=df['qvals']<.001
df.sort_values(by='qvals')

In [ ]:
import itertools
from scipy.stats import false_discovery_control
def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
passages1 = []
passages2=[]
mswitchs = []
pvalues = []
stats = []

for mswitch in ['T','F']:
    df_mswitch = df_abun_get_alpha_only_double.loc[(df_abun_get_alpha_only_double['mswitch']==mswitch)+\
        (df_abun_get_alpha_only_double['passage']==0),:]
    
    for passage_pair in [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(0,7)]:
        p1,p2 = passage_pair
        mswitchs.append(mswitch)
        passages1.append(p1)
        passages2.append(p2)
        x=df_mswitch.loc[df_mswitch['passage']==p1,'counts'].values
        y=df_mswitch.loc[df_mswitch['passage']==p2,'counts'].values
        res = permutation_test((x,y),statistic, vectorized=True,permutation_type='independent',
                           n_resamples=10000, alternative='two-sided')
        pvalues.append(res.pvalue)
        stats.append(res.statistic)
qvals = false_discovery_control(pvalues)
df = pd.DataFrame({'passages1':passages1, 'passages2':passages2, 
                   'mswitchs':mswitchs, 
                   'pval':pvalues,'stats':stats,'qvals':qvals})
df.sort_values(by='qvals')
df['sigs']=df['qvals']<.05
df['sigsig']=df['qvals']<.001
df.sort_values(by='qvals')

In [ ]:
df_abun_get_alpha_only_double.loc[df_abun_get_alpha_only_double['counts']<20,'mesocosm'].unique()

In [ ]:
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
df_abun_get_alpha_only_single = df_abun_get_alpha.loc[df_abun_get_alpha['subjects'].isin(['AA-AA',
                                                                                      'AE-AE','AF-AF']),:]
df_abun_get_alpha_only_single.head()

In [ ]:
df_abun_get_alpha_only_single['coalescence_media']=df_abun_get_alpha_only_single['type_mesocosm'].transform(lambda x: x.split('-')[-1])
df_abun_get_alpha_only_single['parent_media']=df_abun_get_alpha_only_single['type_mesocosm'].transform(lambda x: x.split('-')[-2])
df_abun_get_alpha_only_single['env']=df_abun_get_alpha_only_single['parent_media']+'-'+df_abun_get_alpha_only_single['coalescence_media']
df_abun_get_alpha_only_single['mswitch'] ='F'
df_abun_get_alpha_only_single.loc[df_abun_get_alpha_only_single['parent_media']!=df_abun_get_alpha_only_single['coalescence_media'],'mswitch']='T'

df_abun_get_alpha_only_single=df_abun_get_alpha_only_single.loc[df_abun_get_alpha_only_single['counts']>10,:]#.unique()
p = iqplot.strip(df_abun_get_alpha_only_single.sort_values(by='passage', ascending=True), 
                 q = 'counts', cats = ['passage','mswitch' ], spread='jitter',
                 color_column = 'mswitch',
                palette= bokeh.palettes.Colorblind[7][::-1],
                  width =650,height=200,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,60)
#p.xaxis.axis_label = 'Same Media - Same Subject'
p.yaxis.axis_label = 'Number of Species'
p.legend.location='right'
p.output_backend='svg'
export_plot_pdf(p,'num_sp_overtime_single')
bokeh.io.show(p)